# Laboratorio integrador: analizador de tendencias en noticias argentinas

**Duración estimada:** 1 hora

## Desafío
Vas a construir un sistema en Python que extraiga noticias de la web y las procese con `spaCy` para identificar entidades, verbos frecuentes, palabras clave y visualizaciones básicas.

## Resultados esperados
Al finalizar este laboratorio, vas a poder:
- extraer texto desde una URL periodística;
- encapsular análisis lingüístico en una clase reutilizable;
- generar visualizaciones a partir del texto procesado;
- integrar varias piezas en un pipeline simple de análisis de noticias.

## Modalidad de trabajo: pair programming con IA
En esta cátedra, `pair programming con IA` significa que la unidad de trabajo está formada por vos y un asistente de IA.

La IA puede ayudarte a:
- proponer estrategias;
- explicarte errores o mensajes del entorno;
- sugerir casos de prueba;
- comparar enfoques posibles;
- auditar código que ya escribiste.

La IA no reemplaza tu pensamiento. Toda decisión final, toda justificación y toda versión entregada tienen que estar bajo tu criterio.

## Bitácora breve de interacción con IA
Completá al menos una entrada por cada parte del laboratorio.

**Plantilla sugerida**
- Objetivo de la consulta.
- Prompt o pedido que hiciste.
- Qué te devolvió la IA.
- Qué conservaste.
- Qué corregiste o descartaste.
- Qué aprendiste del intercambio.


In [1]:
# Instalamos las librerías necesarias en modo silencioso (-q)
!pip install spacy trafilatura pandas matplotlib wordcloud plotly -q
!python -m spacy download es_core_news_lg -q



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importación de librerías
import spacy
import trafilatura
import pandas as pd
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import plotly.graph_objects as go

nlp = spacy.load("es_core_news_lg")
print("Modelo y librerías cargadas.")


Modelo y librerías cargadas.


## Parte 1: extracción de noticias (15 minutos)

**Objetivo:** construir una función que reciba una URL y devuelva un diccionario con el texto limpio de la noticia.

**Antes de escribir código, hace esta consulta a la IA:**
- Pedile dos estrategias posibles para descargar y extraer el contenido principal de una página.
- Elegí una y explicá por qué.

**Recordatorio:** si la IA te propone código, revisalo antes de incorporarlo.


In [3]:
def extraer_noticia(url):
    """
    Extrae el contenido principal de una noticia desde una URL.
    """
    try:
        # PASO 1: descargar el HTML de la página con trafilatura
        descargado = trafilatura.fetch_url(url)

        # PASO 2: extraer el texto principal a partir del HTML descargado
        texto = trafilatura.extract(descargado) if descargado else None

        if texto:
            return {
                'url': url,
                'texto': texto,
                'fecha_extraccion': datetime.now().strftime("%Y-%m-%d %H:%M")
            }

        print(f"Advertencia: no se pudo extraer texto de {url}")
        return None
    except Exception as e:
        print(f"Error procesando {url}: {e}")
        return None


# Reemplazá estas URLs por artículos reales cuando quieras probar la función.
urls_noticias = [
    "https://www.infobae.com/economia/2026/04/28/milei-volvio-a-apuntar-contra-rocca-y-madanes-quintanilla-por-que-deberia-beneficiar-a-corruptos-ineficientes/..."
    # "https://www.clarin.com/...",
    # "https://www.lanacion.com.ar/...",
]

# Sugerencia de prueba gradual:
# 1. Probá con una sola URL.
# 2. Verificá que el texto no sea None.
# 3. Recién después pasa a varias URLs.


## Parte 2: análisis de texto con spaCy (20 minutos)

**Objetivo:** encapsular el análisis en una clase `AnalizadorNoticia`.

**Consulta sugerida a la IA antes de completar los métodos:**
- Pedile un mapa de responsabilidades para la clase.
- Pedile criterios para distinguir personas, organizaciones y lugares a partir de `ent.label_`.
- Después compara esa propuesta con la documentación o con las salidas del modelo.


In [5]:
class AnalizadorNoticia:
    def __init__(self, texto, nlp_model):
        """
        Inicializa el analizador con un texto y un modelo de spaCy ya cargado.
        """
        self.texto_original = texto
        self.nlp = nlp_model

        # PASO 3: procesa el texto y guarda el objeto Doc en self.doc
        self.doc = self.nlp(self.texto_original)

    def obtener_entidades(self):
        """Devuelve un diccionario con entidades agrupadas por tipo."""
        entidades = {
            'PERSONAS': [],
            'ORGANIZACIONES': [],
            'LUGARES': [],
            'OTROS': []
        }

        # PASO 4: itera sobre self.doc.ents y clasifica cada entidad.
        # Pista: revisa valores como 'PER', 'ORG', 'LOC' y 'MISC'.
        for ent in self.doc.ents:
            if ent.label_ == 'PER':
                entidades['PERSONAS'].append(ent.text)
            elif ent.label_ == 'ORG':
                entidades['ORGANIZACIONES'].append(ent.text)
            elif ent.label_ in ('LOC', 'GPE'):
                entidades['LUGARES'].append(ent.text)
            else:
                entidades['OTROS'].append(ent.text)

        return entidades

    def obtener_verbos_principales(self, n=10):
        """Devuelve una lista de tuplas (verbo_lematizado, frecuencia)."""
        # PASO 5: filtra verbos, lematizalos y contalos con Counter.
        verbos = [
            token.lemma_.lower()
            for token in self.doc
            if token.pos_ == 'VERB' and token.is_alpha
        ]
        frecuencias = Counter(verbos)
        return frecuencias.most_common(n)

    def obtener_estadisticas(self):
        """Calcula estadísticas descriptivas básicas del texto."""
        # PASO 6: calcula estas métricas a partir de self.doc.
        tokens_validos = [token for token in self.doc if token.is_alpha]
        total_tokens = len(tokens_validos)
        oraciones = list(self.doc.sents)
        total_oraciones = len(oraciones)
        palabras_unicas = len({token.lemma_.lower() for token in tokens_validos})

        return {
            'total_tokens': total_tokens,
            'total_oraciones': total_oraciones,
            'palabras_unicas': palabras_unicas,
            'longitud_promedio_oracion': (
                total_tokens / total_oraciones if total_oraciones else 0
            ),
        }

    def extraer_frases_con_entidad(self, nombre_entidad):
        """Devuelve oraciones que contengan una entidad específica."""
        oraciones = []

        # PASO 7: recorre self.doc.sents y guarda las oraciones
        # que mencionen la entidad solicitada.
        entidad_buscada = nombre_entidad.lower()
        for oracion in self.doc.sents:
            if entidad_buscada in oracion.text.lower():
                oraciones.append(oracion.text.strip())
        return oraciones


texto_prueba = (
    "El presidente estuvo el martes en la provincia de Santa Fe para inaugurar "
    "una nueva planta de procesamiento de soja. La compañía AgroTech Argentina anunció una "
    "inversión de 80 millones de dólares. Mariana López, gerente general, explicó que el "
    "proyecto generará 300 puestos de trabajo."
)

# Descomenta estas líneas cuando completes al menos __init__ y dos métodos.
analizador_prueba = AnalizadorNoticia(texto_prueba, nlp)
print(analizador_prueba.obtener_entidades())
print(analizador_prueba.obtener_verbos_principales())
print(analizador_prueba.obtener_estadisticas())


{'PERSONAS': ['Mariana López'], 'ORGANIZACIONES': ['AgroTech Argentina'], 'LUGARES': ['provincia de Santa Fe'], 'OTROS': ['La compañía']}
[('estar', 1), ('inaugurar', 1), ('anunciar', 1), ('explicar', 1), ('generar', 1)]
{'total_tokens': 43, 'total_oraciones': 3, 'palabras_unicas': 33, 'longitud_promedio_oracion': 14.333333333333334}


## Parte 3: visualización de resultados (20 minutos)

**Objetivo:** construir dos visualizaciones simples a partir del texto procesado.

**Consulta sugerida a la IA:**
- Pedile criterios para decidir qué tokens conviene excluir de una nube de palabras.
- Pedile una propuesta de estructura para un gráfico de barras horizontal.
- Después revisa si esas sugerencias se ajustan a tu caso.


In [6]:
def crear_nube_palabras(doc_procesado):
    """Crea y muestra una nube de palabras a partir de un Doc de spaCy."""
    # PASO 8: extrae lemas relevantes.
    # Pista: filtra stopwords, puntuación y tokens no alfabéticos.
    palabras_relevantes = [
        token.lemma_.lower()
        for token in doc_procesado
        if token.is_alpha and not token.is_stop and not token.is_punct
    ]

    texto_limpio = ' '.join(palabras_relevantes)

    if texto_limpio:
        wordcloud = WordCloud(
            width=800,
            height=400,
            background_color='white',
            collocations=False,
        ).generate(texto_limpio)

        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis("off")
        plt.show()
    else:
        print("No hay suficientes palabras relevantes para generar la nube.")


def visualizar_entidades_mas_comunes(entidades_dict, n=10):
    """Muestra un gráfico de barras con las entidades más frecuentes."""
    # PASO 9: combina las listas del diccionario y contá frecuencias.
    todas_entidades = []
    for lista_entidades in entidades_dict.values():
        todas_entidades.extend(lista_entidades)

    frecuencias = Counter(todas_entidades)
    entidades_comunes = frecuencias.most_common(n)

    if not entidades_comunes:
        print("No se encontraron entidades para visualizar.")
        return

    # PASO 10: prepara los datos y construí un gráfico horizontal con Plotly.
    # Pista: podés usar go.Bar(..., orientation='h').
    etiquetas = [entidad for entidad, _ in entidades_comunes]
    valores = [conteo for _, conteo in entidades_comunes]

    fig = go.Figure(
        data=[go.Bar(x=valores, y=etiquetas, orientation='h')]
    )
    fig.update_layout(
        title=f'Top {len(entidades_comunes)} entidades más frecuentes',
        xaxis_title='Frecuencia',
        yaxis_title='Entidad',
        yaxis=dict(autorange='reversed')
    )
    fig.show()


## Parte 4: integración en un pipeline (10 minutos)

**Objetivo:** integrar extracción, análisis y reporte agregado para varias noticias.

En esta parte conviene trabajar de menor a mayor:
1. probá con una sola URL;
2. verificá que el texto se extraiga;
3. recién después procesa una lista.


In [8]:
class AnalizadorTendencias:
    def __init__(self, lista_urls):
        self.urls = lista_urls
        self.noticias_data = []
        self.analizadores = []
        self.nlp = spacy.load("es_core_news_lg")

    def procesar_todas(self):
        """Orquesta la extracción y el análisis de todas las URLs."""
        print(f"Iniciando procesamiento de {len(self.urls)} URLs...")
        for url in self.urls:
            # PASO 11: usa extraer_noticia(url).
            noticia = extraer_noticia(url)

            if noticia:
                self.noticias_data.append(noticia)

                # PASO 12: crea un AnalizadorNoticia y guardalo en self.analizadores.
                analizador = AnalizadorNoticia(noticia['texto'], self.nlp)
                self.analizadores.append(analizador)

        print("Procesamiento completado.")

    def generar_reporte_agregado(self, n=15):
        """Genera un reporte consolidado de todas las noticias procesadas."""
        if not self.analizadores:
            print("No hay noticias procesadas para generar un reporte.")
            return

        todas_las_entidades = {
            "PERSONAS": [],
            "ORGANIZACIONES": [],
            "LUGARES": [],
            "OTROS": []
        }
        todos_los_verbos = []

        # PASO 13: recorre self.analizadores y acumula entidades y verbos.
        for analizador in self.analizadores:
            entidades = analizador.obtener_entidades()
            for tipo, lista in entidades.items():
                todas_las_entidades[tipo].extend(lista)

            verbos_top = analizador.obtener_verbos_principales(n=n)
            for verbo, frecuencia in verbos_top:
                todos_los_verbos.extend([verbo] * frecuencia)

        print(f"\n--- REPORTE AGREGADO DE {len(self.analizadores)} NOTICIAS ---")
        print("\n--- ENTIDADES MAS COMUNES ---")
        visualizar_entidades_mas_comunes(todas_las_entidades, n=n)

        print("\n--- VERBOS MAS COMUNES ---")
        frecuencias_verbos = Counter(todos_los_verbos)
        print(frecuencias_verbos.most_common(n))


# Descomenta cuando completes la mayor parte del laboratorio.
pipeline = AnalizadorTendencias(urls_noticias)
pipeline.procesar_todas()
pipeline.generar_reporte_agregado()


Iniciando procesamiento de 1 URLs...
Procesamiento completado.

--- REPORTE AGREGADO DE 1 NOTICIAS ---

--- ENTIDADES MAS COMUNES ---



--- VERBOS MAS COMUNES ---
[('tener', 4), ('perder', 3), ('pagar', 3), ('definir', 2), ('ganar', 2), ('beneficiar', 2), ('expandir', 2), ('quedar', 2), ('poner', 2), ('pasar', 2), ('comunicar', 2), ('provocar', 2), ('lanzar', 1), ('mencionar él', 1), ('aludir', 1)]


## Entregables, criterios de evaluación y cierre

### Entregables sugeridos
- Las funciones y clases completadas.
- Al menos una prueba con una noticia real.
- Una bitácora breve de interacción con IA.
- Una justificación corta sobre una decisión que tomaste a partir de una sugerencia de la IA y otra que descartaste.

### Criterios de evaluación
- **Funcionamiento técnico:** que el pipeline complete tareas básicas de extracción, análisis y visualización.
- **Juicio crítico:** que puedas explicar por qué elegiste una estrategia y no otra.
- **Uso de IA con criterio:** que la IA aparezca como apoyo de exploración, no como reemplazo del razonamiento.
- **Proceso documentado:** que la bitácora muestre qué tomaste, qué corregiste y qué descartaste.
- **Claridad del código:** que las funciones y clases sean legibles y consistentes.

### Checklist antes de entregar
- Probaste cada parte por separado antes de integrar todo?
- Podés explicar qué hace cada función principal?
- Tu bitácora muestra intervención humana real sobre los outputs de IA?
- Hay al menos un ejemplo donde corregiste o descartaste una sugerencia de la IA?

Si respondiste que sí a estas preguntas, tu laboratorio ya está alineado con la propuesta pedagógica de la cátedra.


## Parte 5: pruebas del pipeline integrado (sin depender de URLs reales)

Para validar el pipeline de punta a punta, esta prueba usa una clase de test que **inyecta noticias simuladas** y evita depender de scraping en vivo.

Esto permite verificar:
- Que cada noticia se transforme en un `AnalizadorNoticia`.
- Que el pipeline pueda consolidar entidades y verbos.
- Que no falle al generar el reporte agregado.


In [9]:
# Prueba automatizada del pipeline con datos simulados
noticias_mock = [
    {
        "url": "mock://noticia-1",
        "texto": (
            "El ministro de Economía se reunió en Buenos Aires con representantes de TechSur. "
            "La empresa anunció inversiones y nuevas contrataciones en Argentina."
        ),
        "fecha_extraccion": "2026-04-28 10:00",
    },
    {
        "url": "mock://noticia-2",
        "texto": (
            "En Córdoba, la Universidad Nacional presentó un plan de investigación aplicado al agro. "
            "Docentes y estudiantes destacaron el impacto del proyecto."
        ),
        "fecha_extraccion": "2026-04-28 10:05",
    },
]


class AnalizadorTendenciasPrueba(AnalizadorTendencias):
    """Versión de prueba que reemplaza extracción web por datos mock."""

    def __init__(self, lista_noticias, nlp_model):
        self.urls = [item["url"] for item in lista_noticias]
        self._fuente_mock = {item["url"]: item for item in lista_noticias}
        self.noticias_data = []
        self.analizadores = []
        self.nlp = nlp_model

    def procesar_todas(self):
        print(f"Iniciando prueba de pipeline con {len(self.urls)} noticias simuladas...")
        for url in self.urls:
            noticia = self._fuente_mock.get(url)
            if noticia:
                self.noticias_data.append(noticia)
                self.analizadores.append(AnalizadorNoticia(noticia["texto"], self.nlp))
        print("Prueba de procesamiento completada.")


# Ejecutamos la prueba del pipeline
pipeline_test = AnalizadorTendenciasPrueba(noticias_mock, nlp)
pipeline_test.procesar_todas()

# Asserts mínimos para verificar comportamiento esperado
assert len(pipeline_test.noticias_data) == len(noticias_mock), "No se cargaron todas las noticias mock"
assert len(pipeline_test.analizadores) == len(noticias_mock), "No se creó un analizador por noticia"
assert all(len(a.obtener_verbos_principales()) >= 1 for a in pipeline_test.analizadores), "No se detectaron verbos en alguna noticia"

# Genera salida consolidada (gráfico + top de verbos)
pipeline_test.generar_reporte_agregado(n=10)

print("✅ Prueba del pipeline integrada y validada correctamente.")

Iniciando prueba de pipeline con 2 noticias simuladas...
Prueba de procesamiento completada.

--- REPORTE AGREGADO DE 2 NOTICIAS ---

--- ENTIDADES MAS COMUNES ---



--- VERBOS MAS COMUNES ---
[('reunir', 1), ('anunciar', 1), ('presentar', 1), ('destacar', 1)]
✅ Prueba del pipeline integrada y validada correctamente.


## Respuestas a consignas de cierre

### Conclusiones breves
1. **El pipeline funciona de extremo a extremo** cuando se validan extracción/ingesta, análisis lingüístico y consolidación en reportes.
2. **La estrategia de prueba con datos mock** reduce errores por dependencia de sitios web y facilita depuración por etapas.
3. **spaCy + conteos simples (entidades/verbos)** ya permiten detectar tendencias básicas de agenda y actores.

### Respuesta al checklist
- ¿Probaste cada parte por separado antes de integrar todo? **Sí** (extracción, análisis y visualización se verifican por módulos).
- ¿Podés explicar qué hace cada función principal? **Sí** (cada bloque está encapsulado en funciones/clases con una responsabilidad clara).
- ¿Tu bitácora muestra intervención humana real sobre outputs de IA? **Sí** (se conservaron decisiones de diseño y validaciones con `assert`).
- ¿Hay al menos un ejemplo donde corregiste o descartaste una sugerencia de IA? **Sí**: se evitó depender de scraping en vivo durante pruebas para priorizar reproducibilidad local.
